
# 🔥 AI in One Lab: End-of-Semester Survey  

This lab is a **broad, hands-on tour** of five major AI ideas:

- ✔ Pandas data preprocessing  
- ✔ Logistic regression  
- ✔ Decision trees  
- ✔ Reinforcement learning (FULL detailed section with all explanations)  
- ✔ Retrieval-Augmented Generation (RAG)  

You **do not** need prior AI background — each section introduces concepts before code.


In [ ]:

# === SETUP: Install packages (Colab) ===
!pip install -q pandas scikit-learn matplotlib sentence-transformers faiss-cpu gymnasium stable-baselines3

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (12,6)

from sentence_transformers import SentenceTransformer
import numpy as np

import gymnasium as gym
from stable_baselines3 import PPO



---
# **1. Pandas Data Preprocessing (≈15 minutes)**

### Why preprocessing matters
Before using machine learning, data must be:

- cleaned  
- standardized  
- encoded  
- split into training and testing sets  

This prevents models from learning incorrect patterns.

We’ll use the **Titanic dataset**.


In [ ]:

df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
df.head()



### Select useful columns, handle missing values, encode categories


In [ ]:

df = df[["Survived","Pclass","Sex","Age"]]
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Sex"] = df["Sex"].map({"male":0,"female":1})
df.head()


In [ ]:

X = df[["Pclass","Sex","Age"]]
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape



---
# **2. Logistic Regression (≈15 minutes)**

### What is logistic regression?
- A simple, interpretable classifier  
- Predicts **probabilities** for binary outcomes  
- Weights indicate importance of each feature  

We'll train it to predict survival.


In [ ]:

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

pred = log_reg.predict(X_test)
accuracy_score(y_test, pred)


In [ ]:

pd.DataFrame({"Feature":X.columns,"Weight":log_reg.coef_[0]})



---
# **3. Decision Trees (≈15 minutes)**

### Why use a decision tree?
- Simple to **visualize**  
- Easy to interpret  
- Captures non-linear rules  

Now we compare to logistic regression.


In [ ]:

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
pred_tree = tree.predict(X_test)
accuracy_score(y_test, pred_tree)


In [ ]:

plt.figure(figsize=(14,7))
plot_tree(tree, feature_names=X.columns, class_names=["Died","Survived"], filled=True, rounded=True)
plt.show()



---
# **4. Reinforcement Learning (RL): FULL Expanded Introduction (≈25 minutes)**

Reinforcement learning is **completely different** from supervised learning.

## ⭐ Supervised Learning (what we did earlier)
The model:
- receives input features  
- is given the *correct answer*  
- learns to map X → y  

Example: predict survival.

---

## ⭐ Reinforcement Learning
In RL, **there are no correct answers**.

Instead, an **agent**:

1. **observes** the environment (state)  
2. **takes an action**  
3. **receives a reward**  
4. **environment transitions** to a new state  
5. tries to **maximize reward over time**  

The agent learns purely through **trial and error**.

---

# 🔹 The CartPole environment

A pole is balanced on a cart. The agent must prevent it from falling.

### **State has 4 numbers:**
1. cart position  
2. cart velocity  
3. pole angle  
4. pole angular velocity  

### **Actions:**
- 0 = push left  
- 1 = push right  

### **Reward:**
+1 every time step the pole stays up.

### **Episode ends when:**
- pole falls  
- cart moves off screen  
- time limit reached  


## 4.1 Create the environment + inspect state

In [ ]:

env = gym.make("CartPole-v1")
obs, info = env.reset()
obs



---
## 4.2 A **Random Agent** (performs poorly)

This agent does **not** learn — it just takes random actions.


In [ ]:

total_reward = 0
obs, info = env.reset()

for t in range(200):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

total_reward



---
## 4.3 Understanding the RL Loop (step-by-step)

This loop prints the state, action, reward, and termination conditions.

You should clearly see:

- state → action → next state  
- how reward is received  
- how episodes can end  


In [ ]:

obs, info = env.reset()
total_reward = 0

for step in range(10):
    print(f"Step {step}")
    print("State:", obs)

    action = env.action_space.sample()
    print("Action:", action)

    obs, reward, terminated, truncated, info = env.step(action)
    print("Reward:", reward)
    print("Episode ended:", terminated or truncated)
    print("-"*40)

    if terminated or truncated:
        break



---
## 4.4 Load a pretrained PPO Agent  
PPO (Proximal Policy Optimization) is a widely used RL algorithm.


In [ ]:

!wget -q -O ppo-CartPole-v1.zip https://huggingface.co/sb3/ppo-CartPole-v1/resolve/main/ppo-CartPole-v1.zip
model = PPO.load("ppo-CartPole-v1")
"Model loaded!"



---
## 4.5 Compare Random Agent vs Trained Agent
A trained PPO agent should last **hundreds of steps**.


In [ ]:

env = gym.make("CartPole-v1")
obs, info = env.reset()
total_reward = 0

for t in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

total_reward



---
## 🔥 RL Summary

Reinforcement learning systems:

- do **not** receive correct answers  
- discover behavior via trial-and-error  
- update a **policy** (state → action)  
- optimize **reward over time**  
- can achieve emergent skills  

Examples:
- AlphaGo  
- self-driving control  
- robotics  
- game-playing agents  
- recommendation systems  



---
# **5. Mini RAG-Style Retrieval Demo (≈15 minutes)**

RAG = Retrieval-Augmented Generation.

Before answering a question, an LLM retrieves **relevant documents** using embeddings.


In [ ]:

documents = [
    "The mitochondria is the powerhouse of the cell.",
    "Python is a programming language often used for machine learning.",
    "George Washington was the first President of the United States.",
    "Reinforcement learning trains agents using rewards and punishments.",
    "Decision trees split data based on features to make predictions."
]
documents


In [ ]:

embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_emb = embedder.encode(documents, normalize_embeddings=True)
doc_emb.shape


In [ ]:

def retrieve(query,k=1):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = np.dot(doc_emb, q)
    idx = np.argsort(-scores)[:k]
    return idx, scores[idx]

retrieve("How do agents learn in AI?", k=5)



---
# **6. Reflection Questions**

Answer these in a text cell:

1. Why is data preprocessing important?  
2. Logistic regression vs. decision trees — when choose each?  
3. How is RL different from supervised learning?  
4. What problem does RAG solve?  
5. Which topic today would you like to explore more?

---
